### Import Modules

In [1]:
import pandas as pd,numpy as np
from sklearn.svm import SVC,LinearSVC,NuSVC
from sklearn.model_selection import train_test_split,cross_val_score, StratifiedKFold,GridSearchCV
from sklearn.preprocessing import LabelEncoder,StandardScaler,MinMaxScaler,normalize
from sklearn.metrics import accuracy_score,classification_report
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from umap import UMAP
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
import tensorflow as tf
from sklearn.model_selection import KFold
from imblearn.under_sampling import RandomUnderSampler

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-02 10:42:03.834961: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Load Dataset

In [2]:
_data = pd.read_csv("./embeddings_result/embeddings_call_graph_clusters_v4.csv")
data_coderank = pd.read_csv("./embeddings_result/embeddings_call_graph_clusters_v4_coderank.csv")
_data['code_file'] = _data['code_file'].apply(lambda x: x.replace('/root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code-v2/',''))
data_coderank['code_file'] = data_coderank['code_file'].apply(lambda x: x.replace('/root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code-v2/',''))
_data['pattern'] = _data['code_file'].apply(lambda x: x.split('/')[0])
data_coderank['pattern'] = data_coderank['code_file'].apply(lambda x: x.split('/')[0])

data = _data
data = data_coderank

In [23]:
data['pattern'].value_counts()

pattern
Explainable AI (XAI) Techniques                                                                 97
Enhanced User Intent Comprehension with LLMs                                                    97
Tool Use for LLMs                                                                               97
Structured Output & Formatting for LLMs                                                         97
LLM Agent Training & Alignment                                                                  97
LLM Results Evaluation                                                                          97
LLM KV Cache Optimization                                                                       97
Modular LLM Agent Architectures                                                                 97
LLMs for Recommender Systems                                                                    97
LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT    97
Re

In [3]:
data.head()

,code_file,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_759,dim_760,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767,pattern
0,Advanced LLM Prompting/pattern_1.py,-0.634104,-0.369418,0.168330,0.234345,-0.094277,0.465088,0.471025,-0.031221,0.889863,...,-0.704033,0.476058,-0.424835,0.767833,0.886136,0.560142,-0.114798,0.096343,0.263163,Advanced LLM Prompting
1,Advanced LLM Prompting/pattern_10.py,1.370919,-0.819198,-0.360426,-1.251280,-0.526137,0.118881,-0.752482,-0.477260,1.422807,...,0.012588,0.336796,-1.772291,0.704595,-0.329259,-0.252389,0.226630,0.535686,0.500727,Advanced LLM Prompting
2,Advanced LLM Prompting/pattern_11.py,0.728744,0.354183,-0.396259,-0.641893,-0.941455,-0.728828,0.901591,-0.109632,-0.023404,...,1.033639,0.379386,-1.264955,-0.363467,-0.191085,-0.659794,0.049255,0.791801,-0.120603,Advanced LLM Prompting
3,Advanced LLM Prompting/pattern_12.py,-0.958193,-0.362376,-0.616644,1.146603,0.145816,0.198301,0.038843,-1.170772,-0.201782,...,1.466568,-0.012082,0.154564,0.181723,0.482150,0.368036,-0.206501,1.219586,1.291036,Advanced LLM Prompting
4,Advanced LLM Prompting/pattern_13.py,-0.506203,-0.517717,-0.487788,0.918230,0.309301,-0.403503,0.712446,-0.600395,0.388052,...,0.930495,0.647112,-0.332691,0.617454,-0.920227,0.916359,0.326085,1.320728,-0.058612,Advanced LLM Prompting


### Helper Functions

In [4]:
def print_scores(y_test, y_pred,target_names):
    print(classification_report(y_test, y_pred, target_names=target_names))

### Preprocess Data

In [5]:
le = LabelEncoder()
data['pattern_encoded'] = le.fit_transform(data['pattern'])
features_cols = [col for col in data.columns if col not in ['code_file','pattern','pattern_encoded']]

In [6]:
X = data[features_cols]
y = data['pattern_encoded']

# balance the dataset
rus = RandomUnderSampler(random_state=42)
X, y = rus.fit_resample(X, y)

# scale
scaler = StandardScaler()
X = scaler.fit_transform(X)


### Feature Selection

In [7]:
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(f_classif, k=500)
selector.fit(X, y)
X_feature_selected = selector.transform(X)

In [8]:
def test_cross_val(model,X,y):
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2,)
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='f1_weighted',)
    return np.mean(cv_scores)

In [9]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

for k in range(200,768,50):
    selector = SelectKBest(f_classif, k=k)
    selector.fit(X, y)
    X_feature_selected_exp = selector.transform(X)
    svc = SVC(kernel='linear', probability=True, random_state=42)
    
    print(f"Average F1 Weighted Score with k={k}: {test_cross_val(svc,X_feature_selected_exp,y)}")

Average F1 Weighted Score with k=200: 0.5257475225169845
Average F1 Weighted Score with k=250: 0.5302823901493978


KeyboardInterrupt: 

### Dimentionality Reductions

#### PCA

In [10]:
pc = PCA(n_components=450, random_state=1)
X_pca = pc.fit_transform(X)
pc2 = PCA(n_components=450, random_state=1)
X_pca_with_feature_selection = pc2.fit_transform(X_feature_selected)

In [22]:
svc = SVC(kernel='linear', probability=True, random_state=42)
for i in range(200,750,50):
    pca = PCA(n_components=i, random_state=1)
    X_pca_temp = pca.fit_transform(X)
    print(f"PCA with n_components={i}, Explained Variance Ratio: {test_cross_val(svc,X_pca_temp,y)}")

PCA with n_components=200, Explained Variance Ratio: 0.5428984331610751
PCA with n_components=250, Explained Variance Ratio: 0.5448087737166571
PCA with n_components=300, Explained Variance Ratio: 0.548852982745274
PCA with n_components=350, Explained Variance Ratio: 0.5466740102949552
PCA with n_components=400, Explained Variance Ratio: 0.5583005366155857
PCA with n_components=450, Explained Variance Ratio: 0.5553623990197778
PCA with n_components=500, Explained Variance Ratio: 0.5554861899009609
PCA with n_components=550, Explained Variance Ratio: 0.5529233535658806
PCA with n_components=600, Explained Variance Ratio: 0.5515234200862594
PCA with n_components=650, Explained Variance Ratio: 0.5526914222862535
PCA with n_components=700, Explained Variance Ratio: 0.5528021231322484


In [ ]:
svc = SVC(kernel='linear', probability=True, random_state=42)
for i in range(200,500,50):
    pca = PCA(n_components=i, random_state=1)
    X_pca_temp = pca.fit_transform(X_feature_selected)
    print(f"PCA with n_components={i}, Feature Selection+PCA: {test_cross_val(svc,X_pca_temp,y)}")

#### UMAP

In [12]:
umap = UMAP(n_components=450, random_state=1)
X_umap = umap.fit_transform(X)
umap = UMAP(n_components=450, random_state=1)
X_umap_with_feature_selection = umap.fit_transform(X_feature_selected)

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### SVC Exp

In [ ]:
svc_model = SVC(kernel='linear', probability=True, random_state=42)
svc_model.fit(X_train, y_train)

svc_pred = svc_model.predict(X_test)
print_scores(y_test, svc_pred, target_names=le.classes_)

In [ ]:
plt.figure(figsize=(20,6))
sns.countplot(x=le.inverse_transform(svc_pred), order=pd.Series(le.inverse_transform(svc_pred)).value_counts().index)
plt.title('Distribution of Predicted Cluster Labels')
plt.xlabel('Cluster Labels')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
params_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma':[1,0.1,0.01,0.001,'scale','auto']
}

grid = GridSearchCV(SVC(),param_grid=params_grid,refit=True,cv=5,verbose=2)
grid.fit(X_train, y_train)
print("Best Parameters:",grid.best_params_)

In [ ]:
svc_model = SVC(kernel='linear', probability=True, random_state=42)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2,)
svc_cv_scores = cross_val_score(svc_model, X, y, cv=kf, scoring='f1_weighted',)
print(f"SVC Cross-Validation F1 Scores: {svc_cv_scores}")
print(f"SVC Mean CV F1 Score: {np.mean(svc_cv_scores)}")

### Other Model EXP

In [13]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

In [14]:
scores = {}
def model_eval(model,name='',X=X,y=y,tag='original'):
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='f1_weighted')
    if name not in scores:
        scores[name] = {}
    scores[name][tag] = np.mean(cv_scores)
    print(f"Summary of model: ")
    print(f" - {name} Cross-Validation F1 Scores: {cv_scores}")
    print(f" - {name} Mean CV F1 Score: {np.mean(cv_scores)}")

def run_evalutions(X,y,tag='original'):
    models = [
        (KNeighborsClassifier(n_neighbors=2), 'KNN'),
        (RandomForestClassifier(n_estimators=26, random_state=1), 'Random Forest'),
        (DecisionTreeClassifier(random_state=1), 'Decision Tree'),
        (SVC(kernel='linear', probability=True, random_state=42), 'SVC Linear'),
        (SVC(kernel='rbf', probability=True, random_state=42), 'SVC RBF'),
    ]
    for model, name in models:
        model_eval(model, name=name, X=X, y=y,tag=tag)

In [15]:
print("="*50+"\n"+"Evaluations on Original Features:")
run_evalutions(X,y)
print("="*50+"\n"+"Evaluations on Feature Selected Features:")
run_evalutions(X_feature_selected,y,tag='Feature Selected')

Evaluations on Original Features:
Summary of model: 
 - KNN Cross-Validation F1 Scores: [0.45072706 0.41600492 0.46277921 0.39361113 0.42333753]
 - KNN Mean CV F1 Score: 0.4292919677455064
Summary of model: 
 - Random Forest Cross-Validation F1 Scores: [0.38012257 0.36393654 0.41399405 0.35945053 0.38215544]
 - Random Forest Mean CV F1 Score: 0.3799318246095515
Summary of model: 
 - Decision Tree Cross-Validation F1 Scores: [0.21758769 0.21352527 0.24839823 0.18765062 0.24459439]
 - Decision Tree Mean CV F1 Score: 0.22235124233765444
Summary of model: 
 - SVC Linear Cross-Validation F1 Scores: [0.5546409  0.50740102 0.60439096 0.57740061 0.52017712]
 - SVC Linear Mean CV F1 Score: 0.5528021231322484
Summary of model: 
 - SVC RBF Cross-Validation F1 Scores: [0.56436908 0.52409842 0.54567345 0.49964866 0.55387934]
 - SVC RBF Mean CV F1 Score: 0.537533790270509
Evaluations on Feature Selected Features:
Summary of model: 
 - KNN Cross-Validation F1 Scores: [0.45327975 0.40802237 0.46582707

In [16]:
print("="*50+"\n"+"Evaluations on PCA Features:")
run_evalutions(X_pca,y,tag='PCA Features')
run_evalutions(X_pca_with_feature_selection,y,tag='PCA Features with Feature Selection')
print("="*50+"\n"+"Evaluations on UMAP Features:")
run_evalutions(X_umap,y,tag='UMAP Features')
run_evalutions(X_umap_with_feature_selection,y,tag='UMAP Features with Feature Selection')

Evaluations on PCA Features:
Summary of model: 
 - KNN Cross-Validation F1 Scores: [0.45502721 0.41516728 0.46771361 0.40354224 0.42294903]
 - KNN Mean CV F1 Score: 0.4328798718334018
Summary of model: 
 - Random Forest Cross-Validation F1 Scores: [0.26250929 0.35842115 0.3146759  0.27960738 0.2649092 ]
 - Random Forest Mean CV F1 Score: 0.29602458260051623
Summary of model: 
 - Decision Tree Cross-Validation F1 Scores: [0.25039418 0.2765289  0.29211308 0.20996081 0.24036109]
 - Decision Tree Mean CV F1 Score: 0.2538716118925808
Summary of model: 
 - SVC Linear Cross-Validation F1 Scores: [0.55437483 0.5156164  0.61014115 0.56963899 0.52704063]
 - SVC Linear Mean CV F1 Score: 0.5553623990197778
Summary of model: 
 - SVC RBF Cross-Validation F1 Scores: [0.56436908 0.53039454 0.54567345 0.49964866 0.56355505]
 - SVC RBF Mean CV F1 Score: 0.5407281544564826
Summary of model: 
 - KNN Cross-Validation F1 Scores: [0.45327975 0.40744503 0.46582707 0.42345559 0.43786807]
 - KNN Mean CV F1 Scor

In [21]:
print("Evaluation Summary Scores(table):")
score_df = pd.DataFrame(scores).T
score_df

Evaluation Summary Scores(table):


,original,Feature Selected,PCA Features,PCA Features with Feature Selection,UMAP Features,UMAP Features with Feature Selection
KNN,0.429292,0.438032,0.432880,0.437575,0.319614,0.351786
Random Forest,0.379932,0.394603,0.296025,0.279496,0.367246,0.402800
Decision Tree,0.222351,0.241733,0.253872,0.259468,0.307639,0.339065
SVC Linear,0.552802,0.563332,0.555362,0.561518,0.358524,0.372708
SVC RBF,0.537534,0.550231,0.540728,0.550231,0.120940,0.137281
Neural Network,0.565432,0.569136,0.560494,0.561728,0.158025,0.171605


#### Neural Network

In [18]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)
def build_model(input_dim=768):
    model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(input_dim,)),

            tf.keras.layers.Dense(512, activation="relu"),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),

            tf.keras.layers.Dense(256, activation="relu"),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),

            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(0.2),

            tf.keras.layers.Dense(26, activation="softmax") 
    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])

    return model
model = build_model()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=16,
    verbose=0
)
loss, acc = model.evaluate(X_val, y_val)
print("Validation Accuracy:", acc)

2025-12-02 10:45:16.566769: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5741 - loss: 1.6843 
Validation Accuracy: 0.5740740895271301


In [19]:
def cross_validate_nn(X, y):
    k=5
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    val_scores = []

    fold = 1
    X = pd.DataFrame(X)
    y = pd.Series(y)

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    for train_index, val_index in kf.split(X):
        print(f"\n===== Fold {fold} / {k} =====")

        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        # build fresh model for each fold
        model = build_model(input_dim=X.shape[1])

        # train
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=30,
            batch_size=8,
            verbose=0
        )

        # evaluate
        loss, acc = model.evaluate(X_val, y_val, verbose=0)
        val_scores.append(acc)
        print(f"Fold {fold} accuracy: {acc:.4f}")

        fold += 1

    print("\n====== RESULTS ======")
    print("Scores:", val_scores)
    print("Mean accuracy:", np.mean(val_scores))
    return np.mean(val_scores)

In [20]:
scores['Neural Network'] = {}
scores['Neural Network']['original'] = cross_validate_nn(X, y)
scores['Neural Network']['Feature Selected'] = cross_validate_nn(X_feature_selected, y)
scores['Neural Network']['PCA Features'] = cross_validate_nn(X_pca, y)
scores['Neural Network']['PCA Features with Feature Selection'] = cross_validate_nn(X_pca_with_feature_selection, y)
scores['Neural Network']['UMAP Features'] = cross_validate_nn(X_umap, y)
scores['Neural Network']['UMAP Features with Feature Selection'] = cross_validate_nn(X_umap_with_feature_selection, y)


===== Fold 1 / 5 =====
Fold 1 accuracy: 0.5802

===== Fold 2 / 5 =====
Fold 2 accuracy: 0.5741

===== Fold 3 / 5 =====
Fold 3 accuracy: 0.5864

===== Fold 4 / 5 =====
Fold 4 accuracy: 0.5494

===== Fold 5 / 5 =====
Fold 5 accuracy: 0.5370

====== RESULTS ======
Scores: [0.5802469253540039, 0.5740740895271301, 0.5864197611808777, 0.5493826866149902, 0.5370370149612427]
Mean accuracy: 0.565432095527649

===== Fold 1 / 5 =====
Fold 1 accuracy: 0.5370

===== Fold 2 / 5 =====
Fold 2 accuracy: 0.6296

===== Fold 3 / 5 =====
Fold 3 accuracy: 0.5802

===== Fold 4 / 5 =====
Fold 4 accuracy: 0.5370

===== Fold 5 / 5 =====
Fold 5 accuracy: 0.5617

====== RESULTS ======
Scores: [0.5370370149612427, 0.6296296119689941, 0.5802469253540039, 0.5370370149612427, 0.5617284178733826]
Mean accuracy: 0.5691357970237731

===== Fold 1 / 5 =====
Fold 1 accuracy: 0.5988

===== Fold 2 / 5 =====
Fold 2 accuracy: 0.5247

===== Fold 3 / 5 =====
Fold 3 accuracy: 0.5864

===== Fold 4 / 5 =====
Fold 4 accuracy: 0.53

#### KNeighbors Classifier

In [ ]:
model = KNeighborsClassifier(n_neighbors=2)
model_eval(model, name='KNN')

In [ ]:
model = RandomForestClassifier(n_estimators=26, random_state=1)
model_eval(model, name='Random Forest')

In [ ]:
model = DecisionTreeClassifier(random_state=1)
model_eval(model)

### Neural Network Validation

In [ ]:
curated_prediction_raw = """https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_22.py	Enhanced User Intent Comprehension
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_27.py	Enhanced User Intent Comprehension
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_28.py	Enhanced User Intent Comprehension
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_29.py	Enhanced User Intent Comprehension
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_31.py	Enhanced User Intent Comprehension
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_38.py	Enhanced User Intent Comprehension
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/CAAFE/cluster_7.py	Explainable AI (XAI) Techniques
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_33.py	Integrating External Knlowladge
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_34.py	Integrating External Knlowladge
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/BruteForceAI/cluster_1.py	Integrating External Knlowladge
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AgentBench/cluster_7.py	Integrating Knlowladge Graph
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AgentBench/cluster_19.py	LLM Agent Training & Alignment
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AgentBench/cluster_20.py	LLM Agent Training & Alignment
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/CAAFE/cluster_2.py	LLM Code Execution for Precision
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/CAAFE/cluster_5.py	LLM Code Execution for Precision
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/ChatSim/cluster_124.py	LLM Code Execution for Precision
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_2.py	LLM Memory, Knowledge & Adaptation
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/3DOD_thesis/cluster_9.py	Multimodal Reasoning Extensions
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/Autonomous-Driving-in-Carla-using-Deep-Reinforcement-Learning/cluster_3.py	Multimodal Reasoning Extensions
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/ChatSim/cluster_10.py	Multimodal Reasoning Extensions
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/ChatSim/cluster_115.py	Multimodal Reasoning Extensions
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/ChatSim/cluster_12.py	Multimodal Reasoning Extensions
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_3.py	Reranking
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_25.py	Retrieval Augmented Generation(RAG) Optimization, handling hallucinations
https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/AIlice/cluster_26.py	Retrieval Augmented Generation(RAG) Optimization, handling hallucinations"""

In [ ]:
curated_prediction = [line.split('\t') for line in curated_prediction_raw.strip().split('\n')]
curated_prediction = {item[0].replace('https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/', ''): item[1] for item in curated_prediction}

In [ ]:
curated_prediction

In [ ]:
t_data = pd.read_csv("../data/raw/repos/embeddings_call_graph_clusters.csv")
t_data['code_file'] = t_data['cluster_file'].apply(lambda x: x.replace('./result/repo_callgraph_clusters/', ''))
feature_cols = [col for col in t_data.columns if col not in ['cluster_file', 'code_file', 'cluster_label']]
features = t_data[feature_cols]

In [ ]:
t_data

In [ ]:
features = scaler.fit_transform(features)
pca_features = pc.transform(features)

In [ ]:
model = build_model(input_dim=X_pca.shape[1])
model.fit(X_pca, y, epochs=30, batch_size=8, verbose=0)

In [ ]:
pred = model.predict(pca_features)
classes = np.argmax(pred, axis=1)

In [ ]:
t_data['nn_predicted_pattern'] = le.inverse_transform(classes)
t_data['curated_pattern'] = t_data['code_file'].map(curated_prediction)

In [ ]:
fildered_t_data = t_data[~t_data['curated_pattern'].isna()]
fildered_t_data[['code_file', 'nn_predicted_pattern', 'curated_pattern']].to_csv('result/nn_pattern_predictions_vs_curated_patterns.csv', index=False)

In [ ]:
for i in le.inverse_transform(classes):
    print(i)

### Final SVC Model Build and Save

In [ ]:
final_svc_model = SVC(kernel='linear', probability=True, random_state=42)
final_svc_model.fit(X, y)

In [ ]:
pipeline = Pipeline(steps=[('scaler', scaler), ('selector', selector), ('svc', final_svc_model)])

In [ ]:
predictions = pipeline.predict(data[features_cols])
accuracy = accuracy_score(y, predictions)
print(f"Overall Accuracy on entire dataset: {accuracy}")

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(x=le.inverse_transform(predictions), order=pd.Series(le.inverse_transform(predictions)).value_counts().index)
plt.title('Distribution of Predicted Cluster Labels')
plt.xlabel('Cluster Labels')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
joblib.dump(pipeline, './models/svc_classification_pipeline_v2.joblib')
joblib.dump(le, './models/label_encoder_v2.joblib')

### Clasify

In [ ]:
t_data = pd.read_csv("../data/raw/repos/embeddings_call_graph_clusters.csv")
t_data['code_file'] = t_data['cluster_file'].apply(lambda x: x.replace('./result/repo_callgraph_clusters/', ''))

In [ ]:
feature_cols = [col for col in t_data.columns if col not in ['cluster_file', 'code_file', 'cluster_label']]
features = t_data[feature_cols]

In [ ]:
t_X = scaler.transform(features)
t_X = selector.transform(t_X)
predictions = final_svc_model.predict_proba(t_X)
_predictions = final_svc_model.predict(t_X)
predicted_labels = le.inverse_transform(_predictions)

In [ ]:
import json
llm_result = json.load(open('result/predicted_clusters_verification_results.json'))

In [ ]:
np.max(predictions, axis=1)

In [ ]:
t_data['prediction_prob'] = np.max(predictions, axis=1)
t_data['prediction'] = predicted_labels

In [ ]:
prediction_prob_dict = {}
for idx, row in t_data.iterrows():
    prediction_prob_dict[row['code_file']] = (row['prediction_prob'], row['prediction'])

In [ ]:
for item in llm_result.keys():
    llm_result[item]['prediction_prob'] = prediction_prob_dict.get(item, None)[0]
    llm_result[item]['prediction'] = prediction_prob_dict.get(item, None)[1]

In [ ]:
confident_list = []
for item in llm_result.keys():
    confident_list.append((llm_result[item]['prediction_prob'],llm_result[item]['verification_result']['score'],llm_result[item]['verification_result']['is_correct'],llm_result[item]['prediction']))

In [ ]:
confident_list

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

true_points  = [x for x in confident_list if x[2] == True]
false_points = [x for x in confident_list if x[2] == False]

plt.figure(figsize=(10,6))

# sns.scatterplot(
#     x=[x[0] for x in false_points],
#     y=[x[1] for x in false_points],
#     color='red',
#     # alpha=0.2,
#     label='False',
#     zorder=1
# )

sns.scatterplot(
    x=[x[0] for x in true_points],
    y=[x[1] for x in true_points],
    color='blue',
    alpha=1,
    label='True',
    zorder=2
)

plt.title('Prediction Probability vs Verification Score')
plt.xlabel('Prediction Probability')
plt.ylabel('Verification Score')
plt.legend()
plt.show()


In [ ]:
import pandas as pd

df = pd.DataFrame(confident_list, columns=[
    "probability", "number", "is_true", "pattern"
])

patterns = df["pattern"].unique()
counts = df.groupby(["pattern", "is_true"]).size().reset_index(name="count")

plt.figure(figsize=(14,6))

sns.barplot(
    data=counts,
    y="pattern",
    x="count",
    hue="is_true"
)

plt.xticks(rotation=90)
plt.title("True/False Count per Pattern")
plt.tight_layout()
plt.show()